### ANALISI DELLA DISTRIBUZIONE E DELLA CRESCITA DEI GENERI
In questa fase, l'obiettivo è esplorare la composizione tematica del dataset. Nella colonna genres i dati sono stati importati come tipo object (stringhe), contenenti rappresentazioni testuali di liste. Per procedere con un'analisi granulare, è necessario un processo di Data Cleaning e Normalizzazione.

Utilizziamo il modulo ast per convertire le stringhe in oggetti lista Python. Questo passaggio è fondamentale per rendere i dati "comprensibili" agli algoritmi di analisi.

**Normalizzazione**: Poiché un singolo anime può appartenere a più categorie (es. "Action, Adventure"), abbiamo "esploso" le liste in entità singole. Questa operazione evita il bias di considerare combinazioni di generi come etichette uniche, permettendo un conteggio accurato di ogni singola occorrenza.

**Analisi Globale**: Un primo grafico a barre illustrerà la distribuzione completa di tutti i generi per identificare i pilastri del mercato.

**Focus Evolutivo**: Analizzeremo l'andamento temporale dei Top 5 generi per volumi di produzione. Per massimizzare la leggibilità, utilizzeremo intervalli quinquennali (ogni 5 anni), evidenziando i trend di crescita storici dal 1960 al 2023.

In [ ]:
# Comprensione il formato
print("\nFormato colonna genres:")
details_df['genres'].head()

In [ ]:
# Conversione da stringa a lista
genres_cleaned_series = details_df['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else x if isinstance(x, list) else [] )

genres_series = genres_cleaned_series.explode()
genre_counts = genres_series.value_counts()

print("\nGeneri più frequenti:")
print(genre_counts)

# Visualizzazione
genre_counts.plot(kind='bar', title='Distribuzione Generi')
plt.ylabel('Numero di Anime')
plt.show()

genre_counts.head(5).plot(kind='bar', title='Distribuzione Top 5 Generi')
plt.ylabel('Numero di Anime')
plt.show()


### Analisi della Crescita Storica dei Generi (1960 - 2023)
In questa sezione, l'obiettivo è analizzare come la produzione dei 5 generi più popolari si è evoluta nel tempo . Per garantire un'analisi accurata e professionale, abbiamo adottato una strategia di Data Transformation che preserva l'integrità del dataset originale.

- **Preprocessing Temporaneo**: Abbiamo convertito la colonna start_date in oggetti datetime all'interno di variabili locali. L'uso dell'accessore .dt per l'estrazione dell'anno è avvenuto senza modificare permanentemente il DataFrame originale, evitando ridondanze in memoria .

- **Modularità dei Dati**: È stato creato un DataFrame di supporto dedicato esclusivamente a questa visualizzazione . Questo ha permesso di isolare la logica di esplosione dei generi (necessaria essendo la colonna di tipo object) dalla struttura dati principale .

- **Visualizzazione Narrativa**: La suddivisione dell'asse temporale è stata impostata con intervalli di 5 anni. Questa scelta permette di evidenziare i cicli di crescita in modo chiaro e scansionato, rendendo il grafico un "headline" immediato per il lettore.

- **Accuratezza Statistica**: L'analisi copre il periodo dal 1960 al 2023. Abbiamo escluso i dati dal 2024 in poi poiché, trattandosi di un'annualità incompleta, avrebbe mostrato un calo di produzione fittizio non rappresentativo dei reali trend di mercato .

In [ ]:
temp_genres = details_df['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) 
                                         and x.startswith('[') else x if isinstance(x, list) else [])

temp_datetime = pd.to_datetime(details_df['start_date'], errors='coerce') 
temp_years = temp_datetime.dt.year
# Creazione DataFrame di analisi 
df_growth = pd.DataFrame({'year': temp_years,'genre': temp_genres}).explode('genre')

top_5_labels = df_growth['genre'].value_counts().head(5).index.tolist()
df_top_5 = df_growth[df_growth['genre'].isin(top_5_labels)]

# Raggruppamento annuale 
growth_data = df_top_5[(df_top_5['year'] >= 1960) & (df_top_5['year'] <= 2023)]
growth_pivot = growth_data.groupby(['year', 'genre']).size().unstack(fill_value=0)

plt.figure(figsize=(14, 7))
sns.set_style("whitegrid")


sns.lineplot(data=growth_pivot, linewidth=2.5, alpha=0.9, dashes=False)

plt.title('Crescita Storica delle Produzioni Anime (Top 5 Generi)', fontsize=16, fontweight='bold')
plt.xlabel('Anno di Riferimento', fontsize=12)
plt.ylabel('Numero di Produzioni Annuali', fontsize=12)

years_range = range(1960, 2025, 5)
plt.xticks(ticks=years_range, rotation=45)

# Griglia e Legenda
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title='Genere', bbox_to_anchor=(1, 1), loc='upper left')

plt.tight_layout()
plt.show()

### DISTRIBUZIONE VOTI  
Distribuzione basata sullo score dei vari anime

In [ ]:
# Statistiche generali sui voti
print("Distribuzione dello Score:")
display(details_df['score'].describe()) 
# Controlliamo se ci sono voti anomali 
outliers_score = details_df.query('score > 10 or score <= 0')
print(f"Numero di outlier rilevati nello score: {len(outliers_score)}")

if len(outliers_score) > 0:
    display(outliers_score)